# Ridge regression with gradient descent

In [1]:
import warnings
import numpy as np
from sklearn.datasets import load_diabetes
from sklearn.linear_model import Ridge, SGDRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
warnings.simplefilter("ignore")

X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=4)

In [2]:
# One-feature example of Sections 2.3 and 4.1 (100 observations, same data as the figures)
from sklearn.datasets import make_regression
X1, y1 = make_regression(n_samples=100, n_features=1, n_informative=1, noise=20, random_state=13)
A1 = np.insert(X1, 0, 1, axis=1)                      # column of 1s, then x
XtX, Xty = A1.T @ A1, A1.T @ y1
print("XtX =", XtX.round(3).tolist(), " Xty =", Xty.round(3).tolist())
I0 = np.diag([0.0, 1.0])
w = np.array([20.0, -5.0])                            # intercept, slope
grad = XtX @ w - Xty + 100 * I0 @ w
print("gradient =", grad.round(3), " new w =", (w - 0.005 * grad).round(3))
for lam in (0, 100, 250):
    h = np.linalg.eigvalsh(XtX + lam * I0)
    print(f"lambda {lam:3d}: eigenvalues {h.round(1)}, factor 1 - 0.005 h_max = {1 - 0.005 * h.max():.2f}")

XtX = [[100.0, 5.841], [5.841, 87.186]]  Xty = [-66.938, 2412.823]
gradient = [ 2037.734 -3231.94 ]  new w = [ 9.811 11.16 ]
lambda   0: eigenvalues [ 84.9 102.3], factor 1 - 0.005 h_max = 0.49
lambda 100: eigenvalues [ 99.6 187.6], factor 1 - 0.005 h_max = 0.06
lambda 250: eigenvalues [ 99.9 337.3], factor 1 - 0.005 h_max = -0.69


## scikit-learn: SGDRegressor with an L2 penalty

In [3]:
sgd = SGDRegressor(penalty="l2", alpha=0.001, max_iter=500, eta0=0.1,
                   learning_rate="constant", random_state=0)
sgd.fit(X_train, y_train)
print("SGDRegressor test R2", round(sgd.score(X_test, y_test), 4))

SGDRegressor test R2 0.4501


## scikit-learn: Ridge with an iterative solver

In [4]:
ridge = Ridge(alpha=0.001, max_iter=500, solver="sparse_cg").fit(X_train, y_train)
print("Ridge sparse_cg test R2", round(ridge.score(X_test, y_test), 4))

Ridge sparse_cg test R2 0.4625


## From scratch: batch gradient descent

In [5]:
class MyRidgeGD:
    def __init__(self, epochs, learning_rate, alpha):
        self.epochs, self.lr, self.alpha = epochs, learning_rate, alpha

    def fit(self, X, y):
        X = np.insert(X, 0, 1, axis=1)
        w = np.insert(np.ones(X.shape[1] - 1), 0, 0.0)   # intercept 0, slopes 1
        I = np.identity(X.shape[1])
        I[0, 0] = 0                                      # intercept not penalised
        for _ in range(self.epochs):
            grad = X.T @ X @ w - X.T @ y + self.alpha * I @ w
            w = w - self.lr * grad
        self.intercept_, self.coef_ = w[0], w[1:]
        return self

    def predict(self, X):
        return X @ self.coef_ + self.intercept_

mine = MyRidgeGD(epochs=500, learning_rate=0.005, alpha=0.001).fit(X_train, y_train)
print("MyRidgeGD test R2", round(r2_score(y_test, mine.predict(X_test)), 4))
print("intercept", round(mine.intercept_, 2))

MyRidgeGD test R2 0.4738
intercept 150.87


## Too large a learning rate

In [6]:
big = MyRidgeGD(epochs=100, learning_rate=0.006, alpha=0.001).fit(X_train, y_train)
print("largest coefficient after 100 epochs:", f"{np.abs(big.coef_).max():.3g}")

largest coefficient after 100 epochs: 1.54e+04


## Early stopping and ridge where the data overfits

In [7]:
# Where early stopping and the ridge penalty matter: 65 features (the 10 plus squares and products),
# 200 training observations, averaged over 50 random splits
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.preprocessing import PolynomialFeatures, StandardScaler

def gd_test_r2(Xa, ya, Xb, yb, epochs, lr=0.04):
    """Plain batch gradient descent (mean squared error, no penalty) from zero coefficients."""
    w, b = np.zeros(Xa.shape[1]), 0.0
    for _ in range(epochs):
        err = ya - (Xa @ w + b)
        b -= lr * (-2 * err.mean()); w -= lr * (-2 * Xa.T @ err / len(Xa))
    return r2_score(yb, Xb @ w + b)

X_poly = PolynomialFeatures(2, include_bias=False).fit_transform(X)
res = []
for s in range(50):
    Xa, Xb, ya, yb = train_test_split(X_poly, y, train_size=200, random_state=s)
    sc = StandardScaler().fit(Xa); Xa, Xb = sc.transform(Xa), sc.transform(Xb)
    ols = LinearRegression().fit(Xa, ya).score(Xb, yb)
    ridge = RidgeCV(alphas=np.logspace(-2, 4, 25)).fit(Xa, ya).score(Xb, yb)   # alpha chosen on the training data
    Xf, Xv, yf, yv = train_test_split(Xa, ya, test_size=0.25, random_state=100 + s)
    stop = max(range(1, 301, 3), key=lambda e: gd_test_r2(Xf, yf, Xv, yv, e))     # stopping epoch chosen on validation data
    res.append((ols, ridge, gd_test_r2(Xa, ya, Xb, yb, stop)))
r = np.mean(res, axis=0)
print(f"mean test R2: OLS {r[0]:.3f}, ridge (alpha by CV) {r[1]:.3f}, early stopping {r[2]:.3f}")


mean test R2: OLS 0.063, ridge (alpha by CV) 0.436, early stopping 0.403
